# eLCV Battery RUL Hackathon Project

This notebook adapts cycle-level lab-cell data into an e-LCV battery-pack RUL workflow.

## Problem mapping
- Target use case: 35 kWh LFP pack, approximately 350 V, active liquid cooling, 10-90% SoC window.
- Vehicle profile: urban e-LCV, 120-150 km/day, overnight AC charging, ambient 20-45 C.
- Outputs: remaining cycle life, remaining calendar life (months), and remaining usable energy until EOL (80% SOH).

## Assumptions and limitations
- Dataset is cell-level under controlled lab conditions, not pack-level telematics.
- SoC, temperature, and current are estimated from available cycle features using physics-informed proxies.
- Capacity fade is approximated from discharge-time-derived capacity proxy.

## Notebook sections
1. Dataset understanding (condensed EDA)
2. Config and feature engineering for hackathon terms
3. Baseline model and sequence model (LSTM)
4. RUL conversion to months and energy
5. SOH/EOL analysis and sensitivity studies
6. Optional Streamlit integration

In [ ]:
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sns.set_style("whitegrid")
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
@dataclass
class Config:
    # Hackathon constants
    pack_energy_kwh: float = 35.0
    nominal_pack_voltage_v: float = 350.0
    soc_min: float = 0.10
    soc_max: float = 0.90
    eol_soh: float = 0.80
    avg_daily_km_low: float = 120.0
    avg_daily_km_high: float = 150.0

    # Proxy constants based on dataset description and practical assumptions
    nominal_cell_capacity_ah: float = 2.8
    nominal_discharge_c: float = 1.5
    nominal_discharge_current_a: float = 4.2
    reference_temperature_c: float = 25.0

    # Modeling
    test_ratio: float = 0.15
    val_ratio: float = 0.15
    sequence_length: int = 20
    batch_size: int = 128
    lstm_hidden_size: int = 64
    lstm_layers: int = 1
    lstm_dropout: float = 0.0
    lstm_epochs: int = 15
    lstm_lr: float = 1e-3

CFG = Config()
DATA_PATH = '/kaggle/input/battery-remaining-useful-life-rul/Battery_RUL.csv' if os.path.exists('/kaggle/input/battery-remaining-useful-life-rul/Battery_RUL.csv') else 'Battery_RUL.csv'

## Dataset understanding (condensed from original EDA)

In [ ]:
def load_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df

df_raw = load_dataset(DATA_PATH)
print(df_raw.shape)
display(df_raw.head())
display(df_raw.describe(include='all').transpose().head(12))

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df_raw['RUL'], bins=40, kde=True, color='#2E8B57')
plt.title('RUL distribution (cycles)')
plt.xlabel('RUL cycles')
plt.ylabel('Count')
plt.show()

numeric_cols = df_raw.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 7))
sns.heatmap(df_raw[numeric_cols].corr(), cmap='RdBu_r', center=0)
plt.title('Numeric feature correlation heatmap')
plt.show()

## Feature engineering for e-LCV mapping

In [ ]:
def find_col(df: pd.DataFrame, candidates) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f'Missing required column. Tried: {candidates}')

def engineer_features(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    out = df.copy()

    cycle_col = find_col(out, ['Cycle_Index', 'cycle_index', 'Cycle Index'])
    discharge_time_col = find_col(out, ['Discharge Time (s)', 'Discharge Time', 'discharge_time'])
    charge_time_col = find_col(out, ['Charging time (s)', 'Charging time', 'charge_time'])
    max_v_col = find_col(out, ['Max. Voltage Dischar. (V)', 'Max Voltage Dischar (V)', 'max_voltage'])
    min_v_col = find_col(out, ['Min. Voltage Charg. (V)', 'Min Voltage Charg (V)', 'min_voltage'])

    out['cycle_index'] = out[cycle_col].astype(float)

    # Capacity proxy from discharge time and nominal current assumption.
    out['capacity_proxy_ah'] = (out[discharge_time_col] / 3600.0) * cfg.nominal_discharge_current_a

    # Approximate SoC window utilization; clipped to physically valid range.
    out['dod'] = (out['capacity_proxy_ah'] / cfg.nominal_cell_capacity_ah).clip(0.05, 1.0)
    out['soc_end'] = cfg.soc_min
    out['soc_start'] = (out['soc_end'] + out['dod']).clip(cfg.soc_min, cfg.soc_max)

    # C-rate proxy using the implied average current from discharge time.
    avg_current_a = out['capacity_proxy_ah'] / np.maximum(out[discharge_time_col] / 3600.0, 1e-6)
    out['c_rate'] = (avg_current_a / cfg.nominal_cell_capacity_ah).clip(0.1, 4.0)

    out['pack_voltage_proxy_v'] = ((out[max_v_col] + out[min_v_col]) / 2.0) * (cfg.nominal_pack_voltage_v / 3.6)
    out['pack_current_proxy_a'] = out['c_rate'] * (cfg.pack_energy_kwh * 1000.0 / cfg.nominal_pack_voltage_v)

    # Temperature proxy: combines cycle aging trend with charge/discharge timing behavior.
    charge_time_norm = (out[charge_time_col] - out[charge_time_col].mean()) / (out[charge_time_col].std() + 1e-6)
    out['temperature_proxy_c'] = (
        cfg.reference_temperature_c
        + 8.0 * (out['cycle_index'] / out['cycle_index'].max())
        + 2.5 * charge_time_norm
    ).clip(20.0, 45.0)

    out['energy_throughput_kwh'] = out['dod'] * cfg.pack_energy_kwh * (cfg.soc_max - cfg.soc_min)
    out['cumulative_energy_kwh'] = out['energy_throughput_kwh'].cumsum()

    out['soh_proxy'] = out['capacity_proxy_ah'] / out['capacity_proxy_ah'].iloc[:30].median()
    out['soh_proxy'] = out['soh_proxy'].clip(0.5, 1.05)
    out['soh_smooth'] = out['soh_proxy'].rolling(25, min_periods=1).mean()

    return out

df_feat = engineer_features(df_raw, CFG)
display(df_feat[['cycle_index', 'dod', 'c_rate', 'temperature_proxy_c', 'energy_throughput_kwh', 'cumulative_energy_kwh', 'RUL']].head())

In [ ]:
def split_timewise(df: pd.DataFrame, cfg: Config):
    df_sorted = df.sort_values('cycle_index').reset_index(drop=True)
    n = len(df_sorted)
    n_test = int(n * cfg.test_ratio)
    n_val = int(n * cfg.val_ratio)

    train = df_sorted.iloc[: n - n_val - n_test].copy()
    val = df_sorted.iloc[n - n_val - n_test : n - n_test].copy()
    test = df_sorted.iloc[n - n_test :].copy()
    return train, val, test

feature_cols = [
    'cycle_index',
    'dod',
    'c_rate',
    'temperature_proxy_c',
    'pack_current_proxy_a',
    'pack_voltage_proxy_v',
    'energy_throughput_kwh',
    'cumulative_energy_kwh',
    'soh_smooth'
]
target_col = 'RUL'

train_df, val_df, test_df = split_timewise(df_feat, CFG)
X_train, y_train = train_df[feature_cols], train_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## Baseline model (RandomForest)

In [ ]:
def regression_metrics(y_true, y_pred, model_name='model'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {
        'model': model_name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': rmse,
        'R2': r2_score(y_true, y_pred),
    }

baseline = RandomForestRegressor(
    n_estimators=350,
    max_depth=14,
    random_state=42,
    n_jobs=-1
)
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)
baseline_metrics = regression_metrics(y_test, pred_baseline, 'RandomForest baseline')
pd.DataFrame([baseline_metrics])

## Sequence model (LSTM over cycle windows)

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def make_windows(X_df, y_series, seq_len):
    X = X_df.values
    y = y_series.values
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i - seq_len : i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=feature_cols, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)

X_train_seq, y_train_seq = make_windows(X_train_scaled, y_train, CFG.sequence_length)
X_val_seq, y_val_seq = make_windows(X_val_scaled, y_val, CFG.sequence_length)
X_test_seq, y_test_seq = make_windows(X_test_scaled, y_test, CFG.sequence_length)

train_loader = DataLoader(SequenceDataset(X_train_seq, y_train_seq), batch_size=CFG.batch_size, shuffle=True)
val_loader = DataLoader(SequenceDataset(X_val_seq, y_val_seq), batch_size=CFG.batch_size, shuffle=False)
test_loader = DataLoader(SequenceDataset(X_test_seq, y_test_seq), batch_size=CFG.batch_size, shuffle=False)

class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(1)

def train_lstm(model, train_dl, val_dl, epochs, lr):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_state = None
    best_val = float('inf')

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                val_losses.append(criterion(pred, yb).item())

        mean_val = float(np.mean(val_losses))
        if mean_val < best_val:
            best_val = mean_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch + 1}/{epochs} | val_mse={mean_val:.3f}')

    model.load_state_dict(best_state)
    return model

def predict_lstm(model, dataloader):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in dataloader:
            xb = xb.to(device)
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds)

lstm = LSTMRegressor(
    input_size=len(feature_cols),
    hidden_size=CFG.lstm_hidden_size,
    num_layers=CFG.lstm_layers,
    dropout=CFG.lstm_dropout
)
lstm = train_lstm(lstm, train_loader, val_loader, epochs=CFG.lstm_epochs, lr=CFG.lstm_lr)
pred_lstm = predict_lstm(lstm, test_loader)

metrics_lstm = regression_metrics(y_test_seq, pred_lstm, 'LSTM sequence')
metrics_compare = pd.DataFrame([baseline_metrics, metrics_lstm])
display(metrics_compare)

## Convert RUL cycles to km, months, and remaining usable energy

In [ ]:
def cycles_to_km(rul_cycles, km_per_full_cycle=160.0):
    return np.asarray(rul_cycles) * km_per_full_cycle

def cycles_to_months(rul_cycles, avg_daily_km=135.0, km_per_full_cycle=160.0):
    km_remaining = cycles_to_km(rul_cycles, km_per_full_cycle=km_per_full_cycle)
    days = km_remaining / max(avg_daily_km, 1e-6)
    return days / 30.0

def cycles_to_remaining_energy(rul_cycles, pack_energy_kwh=35.0):
    return np.asarray(rul_cycles) * pack_energy_kwh * (CFG.soc_max - CFG.soc_min)

test_results = test_df.copy().reset_index(drop=True)
test_results['rul_cycles_true'] = y_test.values
test_results['rul_cycles_pred_baseline'] = pred_baseline

test_results['RUL_months'] = cycles_to_months(test_results['rul_cycles_pred_baseline'], avg_daily_km=135.0)
test_results['RUL_energy_kwh'] = cycles_to_remaining_energy(test_results['rul_cycles_pred_baseline'], CFG.pack_energy_kwh)
display(test_results[['cycle_index', 'rul_cycles_true', 'rul_cycles_pred_baseline', 'RUL_months', 'RUL_energy_kwh']].head())

## SOH / capacity-fade analysis and EOL crossing

In [ ]:
eol_rows = df_feat[df_feat['soh_smooth'] <= CFG.eol_soh]
eol_cycle = int(eol_rows['cycle_index'].iloc[0]) if len(eol_rows) > 0 else None

plt.figure(figsize=(10, 5))
plt.plot(df_feat['cycle_index'], df_feat['soh_proxy'], alpha=0.35, label='SOH proxy')
plt.plot(df_feat['cycle_index'], df_feat['soh_smooth'], linewidth=2.2, label='SOH smoothed')
plt.axhline(CFG.eol_soh, color='red', linestyle='--', label='EOL SOH (80%)')
if eol_cycle is not None:
    plt.axvline(eol_cycle, color='black', linestyle=':', label=f'EOL cycle ~ {eol_cycle}')
plt.title('Capacity fade / SOH vs cycle')
plt.xlabel('Cycle index')
plt.ylabel('SOH')
plt.legend()
plt.show()

sample_idx = min(100, len(test_results) - 1)
sample_pred_cycles = test_results.loc[sample_idx, 'rul_cycles_pred_baseline']
sample_pred_months = cycles_to_months(sample_pred_cycles, avg_daily_km=135.0)
print(f'Sample predicted remaining cycles to EOL: {sample_pred_cycles:.1f}')
print(f'Sample predicted months to EOL: {sample_pred_months:.2f}')

## Sensitivity analysis (temperature, DoD, C-rate)

In [ ]:
def run_sensitivity(base_row: pd.Series, model, cfg: Config):
    rows = []

    for t in [20, 25, 30, 35, 40, 45]:
        r = base_row.copy()
        r['temperature_proxy_c'] = t
        rows.append(('temperature_proxy_c', t, r))

    for d in [0.30, 0.45, 0.60, 0.75, 0.90]:
        r = base_row.copy()
        r['dod'] = d
        r['energy_throughput_kwh'] = d * cfg.pack_energy_kwh * (cfg.soc_max - cfg.soc_min)
        rows.append(('dod', d, r))

    for c in [0.5, 0.75, 1.0, 1.5, 2.0]:
        r = base_row.copy()
        r['c_rate'] = c
        r['pack_current_proxy_a'] = c * (cfg.pack_energy_kwh * 1000.0 / cfg.nominal_pack_voltage_v)
        rows.append(('c_rate', c, r))

    out = []
    for factor, level, row in rows:
        pred_cycles = float(model.predict(pd.DataFrame([row[feature_cols]]))[0])
        out.append({
            'factor': factor,
            'level': level,
            'rul_cycles': pred_cycles,
            'rul_months': float(cycles_to_months(pred_cycles, avg_daily_km=135.0)),
            'rul_energy_kwh': float(cycles_to_remaining_energy(pred_cycles, cfg.pack_energy_kwh))
        })
    return pd.DataFrame(out)

base_row = test_results.loc[sample_idx, feature_cols]
sens_df = run_sensitivity(base_row, baseline, CFG)
display(sens_df.head())

for factor in ['temperature_proxy_c', 'dod', 'c_rate']:
    sub = sens_df[sens_df['factor'] == factor]
    plt.figure(figsize=(7, 4))
    plt.plot(sub['level'], sub['rul_cycles'], marker='o')
    plt.title(f'Sensitivity of predicted RUL to {factor}')
    plt.xlabel(factor)
    plt.ylabel('Predicted RUL cycles')
    plt.show()

## Hackathon-style visualizations

In [ ]:
# Predicted vs true RUL
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred_baseline, alpha=0.45, label='Baseline')
min_v = min(y_test.min(), pred_baseline.min())
max_v = max(y_test.max(), pred_baseline.max())
plt.plot([min_v, max_v], [min_v, max_v], 'r--')
plt.xlabel('True RUL cycles')
plt.ylabel('Predicted RUL cycles')
plt.title('Predicted vs true RUL')
plt.legend()
plt.show()

# RUL months trend for an example vehicle profile
example_profile = test_results[['cycle_index', 'RUL_months']].sort_values('cycle_index').head(200)
plt.figure(figsize=(10, 4))
plt.plot(example_profile['cycle_index'], example_profile['RUL_months'], color='#1f77b4')
plt.title('Estimated remaining calendar life over cycles (example e-LCV profile)')
plt.xlabel('Cycle index')
plt.ylabel('Remaining life (months)')
plt.show()

## Optional Streamlit dashboard handoff
Run the companion app in app.py after exporting artifacts if needed:
- Save model objects and selected metadata
- Run: streamlit run app.py

In [ ]:
import pickle

artifact = {
    'model': baseline,
    'feature_cols': feature_cols,
    'config': CFG.__dict__,
}

with open('baseline_rul_artifact.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print('Saved baseline_rul_artifact.pkl')

## Hackathon requirement traceability

This section maps each judging requirement to concrete notebook outputs.

- RUL outputs in cycles, months, and energy:
  - Baseline and sequence models predict `RUL` in cycles.
  - Conversion helpers: `cycles_to_km`, `cycles_to_months`, `cycles_to_remaining_energy`.
  - Derived outputs: `RUL_months`, `RUL_energy_kwh`.

- Degradation drivers (temperature, DoD, C-rate, cycles):
  - Engineered features: `temperature_proxy_c`, `dod`, `c_rate`, `cycle_index`, `cumulative_energy_kwh`.
  - Sensitivity block varies each driver and compares resulting RUL.

- SOH/capacity fade and EOL at 80%:
  - `soh_proxy` and `soh_smooth` plotted against cycle index.
  - EOL crossing highlighted with horizontal 80% SOH threshold and cycle marker.

- EV/BMS deployability:
  - `baseline_rul_artifact.pkl` stores model + feature schema + config.
  - This artifact can be loaded by an eLCV BMS service or fleet analytics backend for inference.

## Figure interpretation guide for judges

Use these points as short narration directly below each key chart during demo.

- SOH vs cycle:
  - The smoothed SOH curve shows progressive degradation and the practical EOL boundary at 80%.
  - The first 80% crossing indicates when battery replacement planning should begin.

- Predicted vs true RUL scatter:
  - Concentration near the diagonal indicates good calibration of cycle-life prediction.
  - Large deviations identify operating regimes where additional telemetry would improve robustness.

- Sensitivity curves:
  - Increasing DoD usually reduces RUL most sharply.
  - Higher temperature and higher C-rate generally reduce RUL due to faster degradation stress.
  - These trends justify BMS controls such as charge-rate derating and thermal management.

## Step-by-step prediction walkthrough (representative samples)

This block demonstrates how a fleet engineer would read one prediction:
1. Start from engineered feature inputs.
2. Predict remaining life in cycles.
3. Convert cycles to calendar months and remaining usable energy.
4. Place the sample on the SOH trajectory for EOL context.

In [ ]:
# Use two representative points in test data: early and late in test horizon.
representative_idx = [0, max(0, len(test_results) - 1)]
rep_rows = test_results.iloc[representative_idx].copy()

records = []
for _, r in rep_rows.iterrows():
    x = pd.DataFrame([r[feature_cols]])
    pred_cycles = float(baseline.predict(x)[0])
    pred_months = float(cycles_to_months(pred_cycles, avg_daily_km=135.0))
    pred_energy = float(cycles_to_remaining_energy(pred_cycles, CFG.pack_energy_kwh))

    cycle_val = float(r["cycle_index"])
    nearest_idx = (df_feat["cycle_index"] - cycle_val).abs().idxmin()
    soh_here = float(df_feat.loc[nearest_idx, "soh_smooth"])

    records.append({
        "cycle_index": cycle_val,
        "temperature_proxy_c": float(r["temperature_proxy_c"]),
        "dod": float(r["dod"]),
        "c_rate": float(r["c_rate"]),
        "pred_rul_cycles": pred_cycles,
        "pred_rul_months": pred_months,
        "pred_rul_energy_kwh": pred_energy,
        "soh_at_cycle": soh_here,
        "distance_to_eol_soh": soh_here - CFG.eol_soh,
    })

walkthrough_df = pd.DataFrame(records)
display(walkthrough_df)

# Optional compact text for narration during demo.
for i, row in walkthrough_df.iterrows():
    print(
        f"Sample {i+1}: cycle={row['cycle_index']:.0f}, DoD={row['dod']:.2f}, "
        f"C-rate={row['c_rate']:.2f}, Temp={row['temperature_proxy_c']:.1f}C -> "
        f"RUL={row['pred_rul_cycles']:.1f} cycles, {row['pred_rul_months']:.2f} months, "
        f"{row['pred_rul_energy_kwh']:.1f} kWh remaining; SOH={row['soh_at_cycle']:.3f}."
    )